In [1]:
# Inicialização do Spark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ETL-Postgres-Kafka")
    .master("local[*]")  # mais estável no Jupyter (sem brigar com versões do spark-master)
    .config(
        "spark.jars.packages",
        ",".join([
            # Kafka connector
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0",
            # Driver do Postgres
            "org.postgresql:postgresql:42.7.1"
        ])
    )
    .getOrCreate()
)

print("PySpark/Spark:", spark.version)

PySpark/Spark: 3.5.0


In [2]:
# Leitura do PostgreSQL
df_pg = (
    spark.read
    .format("jdbc")
    .option("url", "jdbc:postgresql://postgres:5432/postgres")
    .option("dbtable", "public.dadostesouroipca")
    .option("user", "postgres")
    .option("password", "postgres")
    .option("driver", "org.postgresql.Driver")
    .load()
)

df_pg.show(5, truncate=False)
df_pg.createOrReplaceTempView("tabela_ipca")

# Consulta SQL sobre dados do Postgres
spark.sql("""
    SELECT Tipo, COUNT(*) AS total
    FROM tabela_ipca
    GROUP BY Tipo
""").show()

+-----------+----------+-------------+------------+-----------+-------------------+-------------------+----+--------------------------+
|CompraManha|VendaManha|PUCompraManha|PUVendaManha|PUBaseManha|Data_Vencimento    |Data_Base          |Tipo|dt_update                 |
+-----------+----------+-------------+------------+-----------+-------------------+-------------------+----+--------------------------+
|8.74       |8.8       |681.12       |677.48      |677.18     |2015-05-15 00:00:00|2005-08-15 00:00:00|IPCA|2026-02-25 16:15:53.446347|
|8.82       |8.9       |310.37       |306.08      |305.94     |2024-08-15 00:00:00|2005-08-15 00:00:00|IPCA|2026-02-25 16:15:53.449422|
|8.74       |8.8       |680.82       |677.18      |676.73     |2015-05-15 00:00:00|2005-08-12 00:00:00|IPCA|2026-02-25 16:15:53.450494|
|8.82       |8.9       |310.23       |305.94      |305.74     |2024-08-15 00:00:00|2005-08-12 00:00:00|IPCA|2026-02-25 16:15:53.451385|
|8.82       |8.9       |310.05       |305.76    

In [3]:
# Leitura do Kafka
df_kafka = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:29092")
    .option("subscribe", "postgres-dadostesouroipca")
    .option("startingOffsets", "earliest")
    .load()
)

# Só pra provar que está lendo sem pesar
df_kafka.selectExpr("topic", "partition", "offset", "timestamp", "length(value) as bytes") \
       .limit(10) \
       .show(truncate=False)

# view pra SQL
df_kafka_str = df_kafka.selectExpr("CAST(value AS STRING) as value")
df_kafka_str.createOrReplaceTempView("mensagens_kafka")

+-------------------------+---------+------+-----------------------+-----+
|topic                    |partition|offset|timestamp              |bytes|
+-------------------------+---------+------+-----------------------+-----+
|postgres-dadostesouroipca|0        |0     |2026-02-25 20:16:42.178|77   |
|postgres-dadostesouroipca|0        |1     |2026-02-25 20:16:42.18 |77   |
|postgres-dadostesouroipca|0        |2     |2026-02-25 20:16:42.181|77   |
|postgres-dadostesouroipca|0        |3     |2026-02-25 20:16:42.181|77   |
|postgres-dadostesouroipca|0        |4     |2026-02-25 20:16:42.181|77   |
|postgres-dadostesouroipca|0        |5     |2026-02-25 20:16:42.182|77   |
|postgres-dadostesouroipca|0        |6     |2026-02-25 20:16:42.182|77   |
|postgres-dadostesouroipca|0        |7     |2026-02-25 20:16:42.182|77   |
|postgres-dadostesouroipca|0        |8     |2026-02-25 20:16:42.182|77   |
|postgres-dadostesouroipca|0        |9     |2026-02-25 20:16:42.183|77   |
+------------------------

In [ ]:
#Encerramento
spark.stop()